<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Do this and restart to get rid of the annoying error

In [1]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cpu
Uninstalling torchvision-0.26.0+cpu:
  Successfully uninstalled torchvision-0.26.0+cpu


Mount the drive

In [1]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Load Dataset

In [2]:
import os
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"

import torch

!pip install datasets
from datasets import load_dataset
from datasets import Dataset


data = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/capstone_usable_qa_data/all_records.json"
)
#print(type(data))
#print(data.keys() if isinstance(data, dict) else "Not a dictionary")
records = data["train"]["All_Records"]

for record in records:
    record["label"] = int(record["label"])
    del record["label"]
dataset = Dataset.from_list(records)

#dataset = Dataset.from_list(data["train"]["All_Records"])
#print (dataset.column_names)

Generating train split: 0 examples [00:00, ? examples/s]

Test, Train, Validation Split

In [3]:
#Test Train Split
from datasets import DatasetDict

# 2. First split: Isolate the final test set (15%)
train_testvalid = dataset.train_test_split(test_size=0.15, seed=42)

# 3. Second split: Split the temporary training set into train (70%) and validation (15%)
test_valid = train_testvalid["train"].train_test_split(test_size=0.176, seed=42)

# 4. Combine everything into a clean DatasetDict
final_dataset = DatasetDict({
    "train": test_valid["train"],
    "validation": test_valid["test"],  # The 'test' of the second split is your validation set
    "test": train_testvalid["test"]
})

#dataset = dataset["All_Records"]["train"].train_test_split(
#    test_size=0.1,
#    seed=42
#)

#train_ds = dataset["train"]
#valid_ds = dataset["test"]

print(len(test_valid["train"]))
print(len(test_valid["test"]))
print(len(train_testvalid["test"]))

12462
2662
2669


Finbert and Tokenizer

In [4]:
#Load Finbert
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

MODEL_NAME = "ProsusAI/finbert"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Tokenize Function

In [5]:
#Tokenize
def tokenize(example):

    return tokenizer(
        example["question"],
        example["reference"],
        truncation=True,
        padding="max_length",
        #padding=False,
        max_length=512
    )

Tokenize

In [6]:
#print(test_valid["train"].column_names)

train_ds = test_valid["train"].map(tokenize, batched=True)
valid_ds = train_testvalid["test"].map(tokenize, batched=True)


"""train_ds = train_ds.remove_columns([
    "question",
    "reference",
    "ticker",
    "section"
])

valid_ds = valid_ds.remove_columns([
    "question",
    "reference",
    "ticker",
    "section"
])"""

Map:   0%|          | 0/12462 [00:00<?, ? examples/s]

Map:   0%|          | 0/2669 [00:00<?, ? examples/s]

'train_ds = train_ds.remove_columns([\n    "question",\n    "reference",\n    "ticker",\n    "section"\n])\n\nvalid_ds = valid_ds.remove_columns([\n    "question",\n    "reference",\n    "ticker",\n    "section"\n])'

In [7]:
train_ds.set_format("torch")
valid_ds.set_format("torch")

Update Column Name For Pytorch

In [8]:
train_ds = train_ds.rename_column("label", "labels")
valid_ds = valid_ds.rename_column("label", "labels")

In [ ]:
#DO NOT RUN
#import torch
#import torchvision

#print("Torch:", torch.__version__)
#print("Torchvision:", torchvision.__version__)



train_ds = train_ds.map( lambda x: {"labels": int(x["label"])})

valid_ds = valid_ds.map(
    lambda x: {"labels": int(x["label"])}
)

In [ ]:
#DO NOT RUN
train_ds = train_ds.remove_columns(["label"])
valid_ds = valid_ds.remove_columns(["label"])
#print(train_ds[0])

In [9]:
columns = [
    "input_ids",
    "attention_mask",
    "token_type_ids",
    "labels"
]


train_ds.set_format(
    type="torch",
    columns=columns
)

valid_ds.set_format(
    type="torch",
    columns=columns
)

Convert Labels To Int

In [10]:
def fix_labels(example):
    return {
        "labels": int(example["labels"])
    }

train_ds = train_ds.map(fix_labels)
valid_ds = valid_ds.map(fix_labels)

Map:   0%|          | 0/12462 [00:00<?, ? examples/s]

Map:   0%|          | 0/2669 [00:00<?, ? examples/s]

DataLoader

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=8
)

batch = next(iter(train_loader))

for k, v in batch.items():
    print(k, type(v))

labels <class 'torch.Tensor'>
input_ids <class 'torch.Tensor'>
token_type_ids <class 'torch.Tensor'>
attention_mask <class 'torch.Tensor'>


In [12]:
import torch
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [13]:
from torch.optim import AdamW
optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

Train The Model

In [ ]:
from tqdm.auto import tqdm
#print(batch)

EPOCHS = 2

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    progress = tqdm(train_loader)

    for batch in progress:
        #for k, v in batch.items():
         # print(k, type(v))
        # Move tensors to GPU
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)

        loss = outputs.loss

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        progress.set_description(
            f"Epoch {epoch+1}"
        )

        progress.set_postfix(
            loss=loss.item()
        )

    print(
        f"Training Loss: {total_loss/len(train_loader):.4f}"
    )

In [26]:
#DO NOT RUN
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

model.eval()

predictions = []
labels = []

with torch.no_grad():

    for batch in valid_loader:

        batch = {
            k: v.to(device)
            for k, v in batch.items()
        }

        outputs = model(**batch)

        pred = outputs.logits.argmax(dim=1)

        predictions.extend(
            pred.cpu().numpy()
        )

        labels.extend(
            batch["labels"].cpu().numpy()
        )

RuntimeError: Expected all tensors to be on the same device, but got index is on cuda:0, different from other tensors on cpu (when checking argument in method wrapper_CUDA__index_select)

In [24]:
accuracy = accuracy_score(
    labels,
    predictions
)

f1 = f1_score(
    labels,
    predictions
)

print("Accuracy:", accuracy)

print("F1:", f1)

Accuracy: 0.27238666167103787
F1: 0.4118715929739552


Save The Model

In [26]:
model.save_pretrained(
    "finbert_relevance_mv"
)

tokenizer.save_pretrained(
    "finbert_relevance_mv"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('finbert_relevance_mv/tokenizer_config.json',
 'finbert_relevance_mv/tokenizer.json')

Load the Saved Model

In [14]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification


tokenizer = AutoTokenizer.from_pretrained(
    "/content/drive/MyDrive/Model_20k_2EP_1IT/finbert_relevance_mv"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "/content/drive/MyDrive/Model_20k_2EP_1IT/finbert_relevance_mv"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [15]:
model = model.to(device)
print("Model device:")
print(next(model.parameters()).device)


Model device:
cpu


In [29]:
model.eval()

predictions = []
probabilities = []
true_labels = []

with torch.no_grad():

    for batch in valid_loader:

        labels = batch["labels"]

        inputs = {
            k: v.to(device)
            for k, v in batch.items()
            if k != "labels"
        }

        outputs = model(**inputs)

        probs = torch.softmax(
            outputs.logits,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        probabilities.extend(
            probs[:,1].cpu().numpy()
        )

        true_labels.extend(
            labels.numpy()
        )

In [28]:
!nvidia-smi

Fri Jul 17 21:41:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             83W /  600W |    1481MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [16]:
print(next(model.parameters()).device)


cuda:0


In [ ]:
import numpy as np

model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in valid_loader:

        # Move tensors to GPU
        batch = {
          k: v.to(device)
          for k, v in batch.items()
            if k != "labels"
        }

        outputs = model(**batch)

        # Get predicted class
        preds = torch.argmax(
            outputs.logits,
            dim=1
        )

        predictions.extend(
            preds.cpu().numpy()
        )

        true_labels.extend(
            batch["labels"].cpu().numpy()
        )


predictions = np.array(predictions)
true_labels = np.array(true_labels)

In [30]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

accuracy = accuracy_score(
    true_labels,
    predictions
)

f1 = f1_score(
    true_labels,
    predictions
)

print("Accuracy:", accuracy)
print("F1 Score:", f1)

print(
    classification_report(
        true_labels,
        predictions
    )
)

Accuracy: 0.7557137504683402
F1 Score: 0.17468354430379746
              precision    recall  f1-score   support

           0       0.76      0.99      0.86      1974
           1       0.73      0.10      0.17       695

    accuracy                           0.76      2669
   macro avg       0.74      0.54      0.52      2669
weighted avg       0.75      0.76      0.68      2669



In [31]:
#DO NOT RUN BUT REFER FOR CONFUSION MATRIX
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
import numpy as np

predictions = trainer.predict(valid_ds)

logits = predictions.predictions
labels = predictions.label_ids

predicted = np.argmax(logits, axis=1)

accuracy = accuracy_score(labels, predicted)

precision, recall, f1, _ = precision_recall_fscore_support(
    labels,
    predicted,
    average="binary"
)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print(confusion_matrix(labels, predicted))

NameError: name 'trainer' is not defined

In [32]:
from sklearn.metrics import roc_auc_score

probabilities = logits[:,1]

auc = roc_auc_score(labels, probabilities)

print("ROC AUC:", auc)

NameError: name 'logits' is not defined

Read Results From Retriever

In [20]:
records = []
with open("/content/drive/MyDrive/capstone_usable_qa_data/retriever_output.txt", "r") as f:
  dict = {}
  i = 0
  #k = 0
  #while k < 10:
  for line in f:
      #print (f"is {i} and line is {line}")
      if i == 1:
        dict["ticker"] = line

      elif i == 2:
        dict["section"] = line

      elif i == 3:
        dict["year"] = line

      elif i == 4:
        dict["reference"] = line
        #print(f"dict {dict}")
        records.append(dict)
        dict = {}
        i = -2
      i = i + 1
    #k = k + 1
  print(f"{records[0]}")

{'ticker': 'NVDA\n', 'section': '1A\n', 'year': '2024\n', 'reference': 'Risk Factors Summary Risks Related to Our Industry and Markets &#8226; Failure to meet the evolving needs of our industry may adversely impact our financial results. &#8226; Competition could adversel\n'}


RE-Rank

In [ ]:
import torch

question = "What supplier risks does NVIDIA face in 2025?"

reranked = []

model.eval()

for chunk in records:
    #print (f"Ref: {chunk["reference"]}")
    inputs = tokenizer(
        question,
        chunk["reference"],
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    # Move tensors to GPU
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probability = torch.softmax(
        logits,
        dim=1
    )[0, 1].item()

    reranked.append({
        "score": probability,
        "chunk": chunk
    })

Sort

In [24]:
reranked.sort(
    key=lambda x: x["score"],
    reverse=True
)

top5 = reranked[:5]

Show Evidence

In [26]:
for r in top5:

    print("=" * 80)

    print("Score:", r["score"])

    print("Ticker:", r["chunk"]["ticker"])

    print("Section:", r["chunk"]["section"])

    print(r["chunk"]["reference"][:400])

Score: 0.631367564201355
Ticker: NVDA

Section: 1A

. These risks may also increase when our products are introduced into new devices, markets, technologies and applications, or new versions are released, and when we rely on partners to supply and manu

Score: 0.6045154929161072
Ticker: NVDA

Section: 1A

Item 1A: Risk Factors The following risk factors could materially and adversely affect our business, financial condition or results of operations and cause reputational harm, and should be carefully c

Score: 0.6045154929161072
Ticker: NVDA

Section: 1A

Item 1A: Risk Factors The following risk factors could materially and adversely affect our business, financial condition or results of operations and cause reputational harm, and should be carefully c

Score: 0.5830297470092773
Ticker: NVDA

Section: 1A

You should consider each of the following factors in evaluating our business and our prospects. However, the risks and uncertainties described below are not the only ones we face. Add

In [27]:
#keep
import os

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = \
    "./triple-mountain-483601-k3-3a823d61bdb7.json"

Create Gemini Prompt

In [32]:
evidence = ""

for i, r in enumerate(top5):

    chunk = r["chunk"]

    evidence += f"""

Evidence {i+1}

Company: {chunk['ticker']}

Section: {chunk['section']}

Confidence: {r['score']:.3f}

{chunk['reference']}

----------------------------------------

"""

prompt = """Question:
What supplier risks does NVIDIA face? Evidence:"""f""" {evidence}""""""Answer using only the evidence above. Cite the section and the company"""

Return Response

In [33]:
#keep
!pip install google-genai

from google import genai

from pydantic import BaseModel
client = genai.Client(
    vertexai=True,
    project="triple-mountain-483601-k3",
    location="us-central1"
    )
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(response.text)

NVIDIA faces supplier risks because it relies on partners to supply and manufacture (NVDA, Section 1A). These risks may also increase when their products are introduced into new devices, markets, technologies and applications, or when new versions are released (NVDA, Section 1A).
